In [ ]:
"""
================================================================================
SECURITY INVESTIGATION PLAYBOOK (Memory Optimized)
================================================================================
Investigate audit events - optimized for large datasets.

Run each PHASE separately if needed to avoid OOM errors.
================================================================================
"""

from pyspark.sql.functions import *

# =============================================================================
# CONFIGURATION
# =============================================================================

EVENT_NAME = "CreatePrivateIp"
PARTITION_DATE = "2026-01-28"

# Limit results to avoid memory issues
MAX_RESULTS = 50

# =============================================================================
# PATHS
# =============================================================================
VOLUME_BASE = "/Volumes/gitrepo/default/git_oci_aidp_silver"
SILVER_AUDIT_PATH = f"{VOLUME_BASE}/audit_logs/data"
SILVER_FLOW_PATH = f"{VOLUME_BASE}/flow_logs/data"

print("=" * 80)
print(f"SECURITY INVESTIGATION: {EVENT_NAME}")
print("=" * 80)
print(f"  Partition: {PARTITION_DATE}")
print("=" * 80)

# =============================================================================
# PHASE 1: INITIAL TRIAGE (lightweight queries first)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 1: INITIAL TRIAGE")
print("=" * 80)

# Load with partition filter - NO CACHE
silver_audit = spark.read.parquet(SILVER_AUDIT_PATH) \
    .filter(col("ingest_date") == PARTITION_DATE)

# 1a. Quick count of target events
print("\n--- 1a. Event Count ---")
target_count = silver_audit.filter(col("event_name") == EVENT_NAME).count()
print(f"Total {EVENT_NAME} events: {target_count:,}")

if target_count == 0:
    print(f"\nNo {EVENT_NAME} events found. Top events:")
    silver_audit.groupBy("event_name").count().orderBy(desc("count")).limit(30).show(truncate=False)
    raise Exception(f"No events for {EVENT_NAME}")

# 1b. Daily volume
print("\n--- 1b. Daily Volume ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .withColumn("event_date", to_date("event_time")) \
    .groupBy("event_date") \
    .agg(count("*").alias("count")) \
    .orderBy("event_date") \
    .show(30, truncate=False)

# 1c. Success/Failure
print("\n--- 1c. Success/Failure ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("response_status") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .show(10, truncate=False)

# =============================================================================
# PHASE 2: ATTRIBUTION
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 2: ATTRIBUTION")
print("=" * 80)

# 2a. Principals
print("\n--- 2a. Principals ---")
principal_df = silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("principal_id", "principal_name", "auth_type") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .limit(MAX_RESULTS)

principal_df.show(MAX_RESULTS, truncate=False)

# Collect principal list (small)
principal_list = [row.principal_id for row in principal_df.select("principal_id").collect() if row.principal_id]
print(f"\nPrincipals to investigate: {principal_list[:10]}...")

# 2b. Source IPs
print("\n--- 2b. Source IPs ---")
ip_df = silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("ip_address") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .limit(MAX_RESULTS)

ip_df.show(MAX_RESULTS, truncate=False)

# Collect IP list (small)
ip_list = [row.ip_address for row in ip_df.select("ip_address").collect() if row.ip_address]
print(f"\nIPs to investigate: {ip_list}")

# =============================================================================
# PHASE 3: CONTEXT - Run these ONE AT A TIME if memory issues persist
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 3: CONTEXT")
print("=" * 80)

# 3a. All events by these principals (top N only)
print("\n--- 3a. Events by Same Principals (Top Events) ---")
if principal_list:
    silver_audit \
        .filter(col("principal_id").isin(principal_list[:5])) \
        .groupBy("principal_name", "event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count")) \
        .limit(MAX_RESULTS) \
        .show(MAX_RESULTS, truncate=False)

# 3b. All events from these IPs
print("\n--- 3b. Events from Same IPs ---")
if ip_list:
    silver_audit \
        .filter(col("ip_address").isin(ip_list)) \
        .groupBy("ip_address", "event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count")) \
        .limit(MAX_RESULTS) \
        .show(MAX_RESULTS, truncate=False)

# 3c. Failures by these actors
print("\n--- 3c. Failed Events ---")
if principal_list or ip_list:
    silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .filter(~col("response_status").isin(["200", "201", "202", "204"])) \
        .groupBy("event_name", "response_status") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count")) \
        .limit(30) \
        .show(30, truncate=False)

# 3d. Timeline (sample only)
print("\n--- 3d. Activity Timeline (Sample) ---")
if principal_list or ip_list:
    silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .select("event_time", "event_name", "principal_name", "ip_address", "response_status") \
        .orderBy("event_time") \
        .limit(50) \
        .show(50, truncate=False)

# =============================================================================
# PHASE 4: IMPACT
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 4: IMPACT")
print("=" * 80)

print("\n--- 4a. Compartments Affected ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("compartment_name") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .limit(20) \
    .show(20, truncate=False)

# =============================================================================
# PHASE 5: CORRELATION (Kill Chain)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 5: CORRELATION")
print("=" * 80)

# 5a. Network events
print("\n--- 5a. Network Changes ---")
if principal_list or ip_list:
    network = silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .filter(
            col("event_name").contains("Security") |
            col("event_name").contains("Route") |
            col("event_name").contains("Gateway") |
            col("event_name").contains("Vcn") |
            col("event_name").contains("Subnet")
        ) \
        .groupBy("event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count"))
    
    if network.count() > 0:
        print("⚠️ Network events found:")
        network.show(20, truncate=False)
    else:
        print("✅ No network changes")

# 5b. IAM events
print("\n--- 5b. IAM Changes ---")
if principal_list or ip_list:
    iam = silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .filter(
            col("event_name").contains("User") |
            col("event_name").contains("Policy") |
            col("event_name").contains("Group") |
            col("event_name").contains("ApiKey")
        ) \
        .groupBy("event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count"))
    
    if iam.count() > 0:
        print("🚨 IAM events found:")
        iam.show(20, truncate=False)
    else:
        print("✅ No IAM changes")

# =============================================================================
# PHASE 6: FLOW LOGS
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 6: FLOW LOGS")
print("=" * 80)

print(f"\nIPs: {ip_list}")

silver_flow = spark.read.parquet(SILVER_FLOW_PATH) \
    .filter(col("ingest_date") == PARTITION_DATE)

# 6a. Traffic FROM IPs
print("\n--- 6a. Traffic FROM IPs ---")
if ip_list:
    flow_from = silver_flow \
        .filter(col("src_ip").isin(ip_list)) \
        .groupBy("src_ip", "dst_ip", "dst_port", "action") \
        .agg(
            count("*").alias("flows"),
            sum("bytes").alias("bytes")
        ) \
        .orderBy(desc("bytes")) \
        .limit(30)
    
    if flow_from.count() > 0:
        flow_from.show(30, truncate=False)
    else:
        print("No outbound flows (IPs may be external API endpoints)")

# 6b. Traffic TO IPs
print("\n--- 6b. Traffic TO IPs ---")
if ip_list:
    flow_to = silver_flow \
        .filter(col("dst_ip").isin(ip_list)) \
        .groupBy("src_ip", "dst_ip", "dst_port", "action") \
        .agg(
            count("*").alias("flows"),
            sum("bytes").alias("bytes")
        ) \
        .orderBy(desc("bytes")) \
        .limit(30)
    
    if flow_to.count() > 0:
        flow_to.show(30, truncate=False)
    else:
        print("No inbound flows found")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"""
Event:        {EVENT_NAME}
Count:        {target_count:,}
Principals:   {len(principal_list)}
IPs:          {len(ip_list)}

IPs Found:    {ip_list}
""")

print("=" * 80)
print("Investigation complete.")
print("=" * 80)

SECURITY INVESTIGATION: CreatePrivateIp
  Partition: 2026-01-28

PHASE 1: INITIAL TRIAGE



--- 1a. Event Count ---


opc-request-id: ACC5E96612F242EB943C448164CF105F/1CAB452795134074B2032A52DF411FA5

Request is cancelled by the user.

In [14]:
# Check folder structure
import os
for item in os.listdir("/Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/data")[:10]:
    print(item)

_SUCCESS
ingest_date=2026-01-28
